# Luma GE workflow for user-defined data

This notebook shows the Luma GE workflow for this user journey:
1. Module 2: User uploads their own classification scheme
2. Module 3: User uploads their own training data
3. Module 6: 
- The system runs `classification.hard_classification()`
- The system computes the model's accuracy based on user's uploaded training data
- The system computes the training data quality as well (Module 4, not shown in this workflow)

# Setup

In [ ]:
import ee 
import luma_ge

service_account_path = '../auth/ee-epstm2024.json'
luma_ge.initialize_with_service_account(service_account_path)

#Check authentication status
status = luma_ge.get_auth_status()
print(f"Initialized: {status['initialized']}")
print(f"Authenticated: {status['authenticated']}")
if status['project']:
    print(f"Project: {status['project']}")

# Module 1

## Upload AOI

In [ ]:
import geemap

aoi = geemap.shp_to_ee('../data/mockup/new_aoi.shp') #change directory

## Satellite imagery retrieval

In [ ]:
from luma_ge.data_acquisition import Reflectance_Data, Reflectance_Stats, final_Image

#Intialize the relfectance class data function
optical_reflectance = Reflectance_Data()
#Initialize the final image class for composite creation
composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2025-01-01'
end = '2025-12-31'
#get the image collection and corresponding statistics
landsat_data, meta = optical_reflectance.get_optical_data(aoi, start, end, optical_data='L9_SR', 
                                                           cloud_cover=40, compute_detailed_stats=False)
#create mosaic between image collection, and clip based on AOI
mosaic_landsat = composite.get_quality_mosaic(landsat_data, aoi, quality_band= 'NDVI', calculate_coverage=False) #REPLACE OLD CODE WITH THE NEW ONE HERE
#Alternatively you can use temporal aggregation (ee reducer) to create mode cloudless imagery
#Add new functionality to calculate the coverage of the composite
median_landsat, coverage = composite.get_temporal_composite(landsat_data, aoi, reducer='Median', calculate_coverage=True)
#retive thermal bands from TOA
thermal_bands, thermal_stats = optical_reflectance.get_thermal_bands(aoi, start, end, cloud_cover=40, thermal_data='L9_TOA', compute_detailed_stats=False)
median_thermal = composite.get_temporal_composite(thermal_bands, aoi, reducer='Median') #REPLACE THE OLD CODE WITH THE NEW ONE
#stacked all landsat bands and convert to float(making sure all data type are compatible)
stacked_landsat = median_landsat.addBands(median_thermal).toFloat()

# Module 2

In [ ]:
import pandas as pd
from luma_ge.classification_scheme import LULC_Scheme_Manager
manager = LULC_Scheme_Manager()

#Temporary function to display the classiifcation scheme in notebook
def display_classification_scheme(manager):
    """Display the current classification scheme in a readable format"""
    if not manager.has_classes():
        print("No classes defined yet.")
        return
    
    print("\n=== Current Classification Scheme ===")
    df = manager.get_dataframe()
    print(df.to_string(index=False))
    
    return df

In [ ]:
# Upload own scheme
csv_path = "../data/mockup/new classification_scheme_template.csv"
df = pd.read_csv(csv_path, sep=None, engine="python")

# Auto-detect columns
id_col, name_col, color_col = manager.auto_detect_csv_columns(df)

# Process CSV upload
manager.process_csv_upload(df, id_col, name_col, color_col)

display_classification_scheme(manager)

LULCTable = df

# Module 3

In [ ]:
import pandas as pd
from luma_ge.sample_data import SyncTrainData

UploadTrainData = True # set as 'true' to upload your own training data shapefile

TrainVectPath  = '../data/mockup/new_training_points.shp'
TrainField = 'ID' 
#         # Load and process training data
TrainDataDict = SyncTrainData.LoadTrainData(
            landcover_df=LULCTable,
            aoi_geometry=aoi,
            training_shp_path=TrainVectPath
        )

TrainDataDict = SyncTrainData.SetClassField(TrainDataDict, TrainField)
TrainDataDict = SyncTrainData.ValidClass(TrainDataDict, 1)
TrainDataDict = SyncTrainData.CheckSufficiency(TrainDataDict, min_samples=20)
TrainDataDict = SyncTrainData.FilterTrainAoi(TrainDataDict)
table_df, total_samples, insufficient_df = SyncTrainData.TrainDataRaw(
    training_data=TrainDataDict.get('training_data'),
    landcover_df=TrainDataDict.get('landcover_df'),
    class_field=TrainDataDict.get('class_field'))
vr = TrainDataDict.get('validation_results', {})

print("=" * 70)
print("TRAINING DATA SUMMARY")
print("=" * 70)
print(f"Total training points loaded     : {vr.get('total_points', 'N/A')}")
print(f"Points after class filtering     : {vr.get('points_after_class_filter', 'N/A')}")
print(f"Valid points (inside AOI)        : {vr.get('valid_points', 'N/A')}")
print(f"Invalid classes found            : {len(vr.get('invalid_classes', []))}")
print(f"Points outside AOI               : {len(vr.get('outside_aoi', []))}")
print("=" * 70)

    # --- Display the main table ---
if table_df is not None and not table_df.empty:
        display_df = table_df.copy()
        if 'Percentage' in display_df.columns:
            display_df['Percentage'] = display_df['Percentage'].apply(
                lambda x: f"{x:.2f}%" if isinstance(x, (int, float)) else x
            )
        display(display_df)
else:
        print("No valid training data available to display.")

TrainDataFinal = TrainDataDict.get('training_data')
labeled_roi = geemap.gdf_to_ee(TrainDataFinal)

# Module 6

In [ ]:
from luma_ge.classification import FeatureExtraction, Generate_LULC
classifier = Generate_LULC()

#Perform Training Test Split
features = FeatureExtraction()
strafied_train, stratified_test = features.stratified_split(labeled_roi, stacked_landsat, 
                            class_prop='kelas', train_ratio=0.7)

# Train model
classification_map, trained_model = classifier.hard_classification(strafied_train, class_property='kelas', image=stacked_landsat,
                                                          ntrees=300, min_leaf=2, return_model=True)

accuracy_metrics = classifier.evaluate_model(
        trained_model=trained_model,
        test_data=stratified_test,
        class_property='kelas'
    )

## Model Accuracy metrics

In [ ]:
# Display accuracy results
print("=== Model Performance Summary ===")
print(f"Overall Accuracy: {accuracy_metrics['overall_accuracy']:.4f} ({accuracy_metrics['overall_accuracy']*100:.2f}%)")
print(f"Kappa Coefficient: {accuracy_metrics['kappa']:.4f}")
print(f"Overall G-Mean: {accuracy_metrics['overall_gmean']:.4f}")

print("\n=== Per-Class Metrics ===")
#Class Dataframe
metrics_df = pd.DataFrame({
    'Precision': accuracy_metrics['precision'],
    'Recall': accuracy_metrics['recall'],
    'F1-Score': accuracy_metrics['f1_scores'],
    'G-Mean': accuracy_metrics['gmean_per_class']
})

# Round to 4 decimal places
metrics_df = metrics_df.round(4)

display(metrics_df)

## Class statistics

In [ ]:
orig_hist = classification_map.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi,
    scale=30,
    maxPixels=1e13
).getInfo()

print(orig_hist)

## Map visualization

In [ ]:
# === Load Classification Scheme ===
scheme = LULCTable
classes = [str(x).strip() for x in scheme["LULC_Type"].tolist()]
palette = [str(x).strip() for x in scheme["Color Palette"].tolist()]
ids = scheme["ID"].tolist()
legend_dict = dict(zip(classes, palette))

# === Visualization Parameters ===
vis_params = {
    "min": min(ids),
    "max": max(ids),
    "palette": palette
}

# === Create geemap Map ===
Map = geemap.Map() 
Map.centerObject(aoi, 7)
Map.addLayer(classification_map, vis_params, "LULC Classification")

# # === Add Legend ===
Map.add_legend(
    title="Land Cover Classification", 
    legend_dict=legend_dict
    )

# Display
Map